# Set up in cloud

In [ ]:
!gsutil -m cp -r gs://seer-models/models/issue_grouping_v1 .

In [ ]:
!gsutil -m -o GSUtil:check_hashes=never cp -r gs://seer-models/models/issue_grouping_v1 issue_grouping_v1/embeddings/model.safetensors

In [ ]:
!gsutil -m cp -r gs://grouping-data/final_csvs .

Add a Colab secret for your `GITHUB_TOKEN`

In [ ]:
from google.colab import userdata
import os

os.environ["GITHUB_TOKEN"] = userdata["GITHUB_TOKEN"]

In [ ]:
!pip install git+https://${GITHUB_TOKEN}@github.com/getsentry/grouping-trainer.git

# Set up

In [1]:
import math

from datasets import Dataset, DatasetDict
import polars as pl
from sentence_transformers import SentenceTransformerTrainingArguments
from sentence_transformers.evaluation import BinaryClassificationEvaluator
from sentence_transformers.training_args import MultiDatasetBatchSamplers
import torch

import grouping_trainer as gt

Some vars to care about

In [2]:
SAMPLE_TRAIN: int | None = None if torch.cuda.is_available() else 30
SAMPLE_VAL: int | None = None if torch.cuda.is_available() else 10

OUTPUT_DIR = "./output"
PER_DEVICE_TRAIN_BATCH_SIZE = 2  # TODO: higher
PER_DEVICE_EVAL_BATCH_SIZE = 2  # TODO: higher
EVAL_STEPS = 5  # TODO: higher

# Load model

In [3]:
# model_path = "issue_grouping_v1/embeddings"
model_path = "/Users/kdubey/projects/seer/models/issue_grouping_v1/embeddings"
model = gt.utils.SentenceTransformer(str(model_path), trust_remote_code=True)
model.device

/Users/kdubey/projects/grouping-trainer/.venv/lib/python3.13/site-packages/torch/onnx/_internal/registration.py:162: OnnxExporterWarning: Symbolic function 'aten::scaled_dot_product_attention' already registered for opset 14. Replacing the existing function with new function. This is unexpected. Please report it on https://github.com/pytorch/pytorch/issues.
  warnings.warn(


device(type='mps', index=0)

In [4]:
assert "layernorm" in repr(model[0].auto_model).lower()
assert "batch" not in repr(model[0].auto_model).lower()

Don't have batch norm. That could mess up stuff for the deduplication strategy.

In [5]:
_ = model.encode("test")

# Load data

In [6]:
def _test_df(df: pl.DataFrame):
    assert df["label"].is_in(["GROUP", "SEPARATE"]).mean() == 1
    assert df["project_id"].is_null().sum() == 0
    assert (
        df.select(
            pl.col("query_stacktrace_string", "candidate_stacktrace_string").fill_null("").str.len_chars().gt(0).all()
        )
        .select(pl.all_horizontal(pl.all()))  # reduce over columns
        .item()
    )
    return gt.utils.deduplicate_pairs(df)


def load_val_df(path: str = "final_csvs/val.csv", sample_size: int | None = None):
    df = pl.read_csv(path)
    df = _test_df(df)

    if sample_size is not None:
        df = df.sample(n=sample_size, seed=42)

    return df


def load_train_df(
    paths: tuple[str, ...] = ("final_csvs/train.csv", "final_csvs/synthetic-semi-easy-negatives.csv"),
    sample_size: int | None = None,
):
    df = gt.utils.concat_vertical_unordered((pl.read_csv(path) for path in paths), how="vertical_relaxed")
    df = _test_df(df)
    assert (df.filter(pl.col("confidence_score").is_null())["source"] == "synthetic-negative-semi-easy").all()

    if sample_size is not None:
        df = df.sample(n=sample_size, seed=42)

    return df


def load_train_dataset_dict(sample_size: int | None = None) -> tuple[DatasetDict, float]:
    df = load_train_df(sample_size=sample_size)
    dataset_dict_train = gt.train.create_project_dataset_dict(df, min_dataset_size=PER_DEVICE_TRAIN_BATCH_SIZE)
    frac_positive = (df["label"] == "GROUP").mean()
    return dataset_dict_train, frac_positive

We'll make a `dataset_val` for val loss.

In [ ]:
dataset_val = gt.train.df_to_dataset(load_val_df(sample_size=SAMPLE_VAL))

In [8]:
dataset_dict_train, frac_positive = load_train_dataset_dict(sample_size=SAMPLE_TRAIN)

  0%|          | 0/26 [00:00<?, ?it/s]

# Set up `Trainer`

In [ ]:
evaluator = BinaryClassificationEvaluator(
    sentences1=list(dataset_val["query_stacktrace_string"]),
    sentences2=list(dataset_val["candidate_stacktrace_string"]),
    labels=[int(record["label"]) for record in dataset_val],
    similarity_fn_names=["cosine"],
    name="val",
    show_progress_bar=True,
    batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
)

Before training:

In [18]:
evaluator(model)

Batches:   0%|          | 0/198 [00:00<?, ?it/s]

{'val_cosine_accuracy': 0.755,
 'val_cosine_accuracy_threshold': 0.9318227767944336,
 'val_cosine_f1': 0.85459940652819,
 'val_cosine_f1_threshold': 0.9134621620178223,
 'val_cosine_precision': 0.7578947368421053,
 'val_cosine_recall': 0.9795918367346939,
 'val_cosine_ap': 0.9071100837411739,
 'val_cosine_mcc': 0.22612357862174923}

In [13]:
def init_bias(frac_positive: float):
    return math.log(frac_positive / (1 - frac_positive))

In [ ]:
trainer = gt.train.Trainer(
    model=model,
    args=SentenceTransformerTrainingArguments(
        # These should prolly be unchanged
        output_dir=OUTPUT_DIR,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=False,
        dataloader_pin_memory=torch.cuda.is_available(),
        num_train_epochs=1,
        #
        # Datalaoder
        multi_dataset_batch_sampler=MultiDatasetBatchSamplers.PROPORTIONAL,
        # Each iter, pick a project randomly, sample from it.
        # Next iter, pick another project randomly, sample from it, etc.
        per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
        seed=42,  # passed to batch sampler
        #
        # Optimizer
        gradient_accumulation_steps=1,
        learning_rate=2e-5,
        learning_rate_mapping={
            # These are important to tune. Higher so that training doesn't get stuck. TODO: check
            r"^log_scale$": 2e-4,
            r"^bias$": 2e-4,
        },
        weight_decay=0.01,
        warmup_ratio=0.1,
        #
        # Eval
        per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        #
        # Logging
        logging_strategy="steps",
        logging_steps=EVAL_STEPS,  # train loss alongside metrics table
        #
        # Checkpointing
        save_strategy="steps",
        save_steps=EVAL_STEPS // 5,
        save_total_limit=2,
    ),
    #
    # Training
    loss=gt.train.SigmoidPairwiseLoss(
        model,
        bias_init=init_bias(frac_positive),
        matryoshka_dims=[768, 512, 256, 128, 64],
        matryoshka_weights=[2, 1, 1, 0.5, 0.25],
        n_dims_per_step=2,
    ),
    data_collator=gt.train.DefaulDataCollator(tokenize_fn=model.tokenize),
    train_dataset=dataset_dict_train,
    shuffle_within_dataset=False,  # more cache hits in each forward
    #
    # Evaluator
    eval_dataset=dataset_val,  # val loss
    evaluator=evaluator,  # val auc, etc.
    #
    # Other
    # callbacks=[...],  # TrainerCallbacks
)

In [17]:
train_output = trainer.train()

Step,Training Loss,Validation Loss,Val Cosine Accuracy,Val Cosine Accuracy Threshold,Val Cosine F1,Val Cosine F1 Threshold,Val Cosine Precision,Val Cosine Recall,Val Cosine Ap,Val Cosine Mcc
5,1.533100,1.555251,0.950000,0.810252,0.967742,0.810252,0.937500,1.000000,0.991389,0.866025
10,0.204700,1.306236,0.950000,0.852326,0.965517,0.852326,1.000000,0.933333,0.992157,0.881917
15,1.393300,1.358383,0.950000,0.854477,0.965517,0.854477,1.000000,0.933333,0.988889,0.881917
20,1.393800,1.389868,0.950000,0.849207,0.965517,0.849207,1.000000,0.933333,0.988889,0.881917
25,3.306700,1.427348,0.950000,0.845443,0.965517,0.845443,1.000000,0.933333,0.988889,0.881917


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]

In [18]:
train_output

TrainOutput(global_step=25, training_loss=1.5663200426101684, metrics={'train_runtime': 44.6034, 'train_samples_per_second': 0.673, 'train_steps_per_second': 0.56, 'total_flos': 0.0, 'train_loss': 1.5663200426101684, 'epoch': 1.0})

In case we OOM'd, consider:

In [ ]:
trainer.per_device_token_budget /= 2
trainer.train(resume_from_checkpoint=True)